In [1]:
# Cell 1: Install dependencies
!pip install ultralytics pyyaml

In [2]:
import os
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────
# Upload your NepaliHTR folder (with train/ and test/ subdirs) to Lightning Studio
# at this path, or adjust accordingly.
SRC_DIR = "/teamspace/studios/this_studio/NepaliHTR"
DATASET_OUT = "/teamspace/studios/this_studio/nepali_htr_yolo"

# Single class
CLASS_NAMES = ["text"]
CLASS_MAP = {"text": 0}

# Split ratios (original train → train + val)
VAL_RATIO = 0.2
RANDOM_SEED = 42

print(f"Source:  {SRC_DIR}")
print(f"Output:  {DATASET_OUT}")
print(f"Classes: {CLASS_NAMES}")

Source:  /teamspace/studios/this_studio/NepaliHTR
Output:  /teamspace/studios/this_studio/nepali_htr_yolo
Classes: ['text']


In [ ]:
# ── Verify source dataset ─────────────────────────────────────────────────
train_dir = os.path.join(SRC_DIR, "train")
test_dir = os.path.join(SRC_DIR, "test")

assert os.path.isdir(train_dir), f"Missing train dir: {train_dir}"
assert os.path.isdir(test_dir), f"Missing test dir: {test_dir}"

train_xmls = sorted([f for f in os.listdir(train_dir) if f.endswith(".xml")])
test_xmls = sorted([f for f in os.listdir(test_dir) if f.endswith(".xml")])

print(f"Train XMLs: {len(train_xmls)}")
print(f"Test XMLs:  {len(test_xmls)}")

Train XMLs: 765
Test XMLs:  193


In [4]:
# ── Conversion helpers ────────────────────────────────────────────────────

def convert_bbox_to_yolo(bbox, img_width, img_height):
    """Convert (xmin, ymin, xmax, ymax) to YOLO (x_center, y_center, w, h) normalized."""
    xmin, ymin, xmax, ymax = bbox
    x_center = (xmin + xmax) / 2.0 / img_width
    y_center = (ymin + ymax) / 2.0 / img_height
    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height
    return (
        max(0.0, min(1.0, x_center)),
        max(0.0, min(1.0, y_center)),
        max(0.0, min(1.0, width)),
        max(0.0, min(1.0, height)),
    )


def parse_voc_annotation(xml_file):
    """Parse Pascal VOC XML and return list of YOLO-format annotation strings."""
    tree = ET.parse(xml_file)
    root = tree.getroot()
    size = root.find("size")
    img_w = int(size.find("width").text)
    img_h = int(size.find("height").text)

    annotations = []
    for obj in root.findall("object"):
        class_name = obj.find("name").text
        if class_name not in CLASS_MAP:
            continue
        class_id = CLASS_MAP[class_name]

        bbox = obj.find("bndbox")
        xmin = max(0, min(float(bbox.find("xmin").text), img_w))
        ymin = max(0, min(float(bbox.find("ymin").text), img_h))
        xmax = max(0, min(float(bbox.find("xmax").text), img_w))
        ymax = max(0, min(float(bbox.find("ymax").text), img_h))

        if xmax <= xmin or ymax <= ymin:
            continue

        xc, yc, w, h = convert_bbox_to_yolo((xmin, ymin, xmax, ymax), img_w, img_h)
        annotations.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

    return annotations


def find_image(directory, stem):
    """Find image file matching the stem (.jpg, .jpeg, .png)."""
    for ext in [".jpg", ".jpeg", ".png"]:
        path = os.path.join(directory, stem + ext)
        if os.path.exists(path):
            return path
    return None


print("Conversion functions defined.")

Conversion functions defined.


In [5]:
# ── Create train / val / test splits and convert ─────────────────────────

output_dir = Path(DATASET_OUT)
if output_dir.exists():
    shutil.rmtree(output_dir)
    print(f"Removed old output directory: {DATASET_OUT}")

for split in ["train", "val", "test"]:
    (output_dir / split / "images").mkdir(parents=True)
    (output_dir / split / "labels").mkdir(parents=True)

# Split original train → train + val
random.seed(RANDOM_SEED)
random.shuffle(train_xmls)
val_count = int(len(train_xmls) * VAL_RATIO)
val_xmls = train_xmls[:val_count]
final_train_xmls = train_xmls[val_count:]

splits = {
    "train": (final_train_xmls, train_dir),
    "val":   (val_xmls, train_dir),
    "test":  (test_xmls, test_dir),
}

print(f"Split sizes → Train: {len(final_train_xmls)}, Val: {len(val_xmls)}, Test: {len(test_xmls)}")

total_images = 0
total_annotations = 0

for split_name, (xml_list, src_dir) in splits.items():
    print(f"\nProcessing {split_name}...")
    skipped = 0
    for xml_file in xml_list:
        stem = os.path.splitext(xml_file)[0]
        xml_path = os.path.join(src_dir, xml_file)

        # Find corresponding image
        img_path = find_image(src_dir, stem)
        if img_path is None:
            skipped += 1
            continue

        # Parse annotations
        annotations = parse_voc_annotation(xml_path)

        # Copy image
        img_ext = os.path.splitext(img_path)[1]
        shutil.copy2(img_path, output_dir / split_name / "images" / f"{stem}{img_ext}")

        # Write YOLO label
        label_path = output_dir / split_name / "labels" / f"{stem}.txt"
        with open(label_path, "w") as f:
            f.write("\n".join(annotations))
            if annotations:
                f.write("\n")

        total_images += 1
        total_annotations += len(annotations)

    print(f"  {split_name}: {len(xml_list) - skipped} converted, {skipped} skipped (no image)")

print(f"\nTotal images: {total_images} | Total annotations: {total_annotations}")

Removed old output directory: /teamspace/studios/this_studio/nepali_htr_yolo
Split sizes → Train: 612, Val: 153, Test: 193

Processing train...
  train: 612 converted, 0 skipped (no image)

Processing val...
  val: 153 converted, 0 skipped (no image)

Processing test...
  test: 193 converted, 0 skipped (no image)

Total images: 958 | Total annotations: 78159


In [6]:
# ── Verify output structure ───────────────────────────────────────────────
for split in ["train", "val", "test"]:
    n_imgs = len(os.listdir(output_dir / split / "images"))
    n_lbls = len(os.listdir(output_dir / split / "labels"))
    print(f"{split:6s} → images: {n_imgs}, labels: {n_lbls}")
    assert n_imgs == n_lbls, f"Mismatch in {split}!"

# Show a sample label
sample_label = sorted(os.listdir(output_dir / "train" / "labels"))[0]
print(f"\nSample label ({sample_label}):")
with open(output_dir / "train" / "labels" / sample_label) as f:
    for line in f.readlines()[:5]:
        print(f"  {line.strip()}")

train  → images: 612, labels: 612
val    → images: 153, labels: 153
test   → images: 193, labels: 193

Sample label (1.txt):
  0 0.088841 0.038029 0.151679 0.043215
  0 0.291983 0.044944 0.159263 0.044944
  0 0.463164 0.050994 0.141928 0.044944
  0 0.617010 0.052723 0.126761 0.053587
  0 0.785482 0.060501 0.112676 0.041487


In [7]:
# ── Create data.yaml ──────────────────────────────────────────────────────
import yaml

data_yaml = {
    "path": DATASET_OUT,
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = os.path.join(DATASET_OUT, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f"Saved: {yaml_path}\n")
print(open(yaml_path).read())

Saved: /teamspace/studios/this_studio/nepali_htr_yolo/data.yaml

path: /teamspace/studios/this_studio/nepali_htr_yolo
train: train/images
val: val/images
test: test/images
nc: 1
names:
- text



In [8]:
# ── Train YOLO11n ─────────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="nepali_htr_yolo11n",
    patience=20,
)

New https://pypi.org/project/ultralytics/8.4.54 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.53 🚀 Python-3.12.11 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/nepali_htr_yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.

In [9]:
# ── Validate on test set ──────────────────────────────────────────────────
metrics = model.val(data=yaml_path, split="test")
print(f"mAP50   : {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.53 🚀 Python-3.12.11 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 601.5±253.5 MB/s, size: 2318.0 KB)
val: Scanning /teamspace/studios/this_studio/nepali_htr_yolo/test/labels... 193 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 193/193 393.2it/s 0.5s0.1s
val: /teamspace/studios/this_studio/nepali_htr_yolo/test/images/130.jpg: corrupt JPEG restored and saved
val: /teamspace/studios/this_studio/nepali_htr_yolo/test/images/595.jpg: corrupt JPEG restored and saved
val: /teamspace/studios/this_studio/nepali_htr_yolo/test/images/596.jpg: corrupt JPEG restored and saved
val: /teamspace/studios/this_studio/nepali_htr_yolo/test/images/597.jpg: corrupt JPEG restored and saved
val: /teamspace/studios/this_studio/nepali_htr_yolo/test/images/598.jpg: corrupt JPEG restored and saved
val: /teamspace/studios/this_studio/nepali_htr_yolo/t

In [10]:
# ── Inference on a sample test image ──────────────────────────────────────
sample_image = os.path.join(DATASET_OUT, "test", "images",
                            os.listdir(os.path.join(DATASET_OUT, "test", "images"))[0])

results = model.predict(
    source=sample_image,
    conf=0.4,
    save=True,
    save_txt=True,
    name="nepali_htr_predict",
)


image 1/1 /teamspace/studios/this_studio/nepali_htr_yolo/test/images/600.jpg: 640x384 19 texts, 96.4ms
Speed: 3.0ms preprocess, 96.4ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 384)
Results saved to /teamspace/studios/this_studio/runs/detect/nepali_htr_predict
1 label saved to /teamspace/studios/this_studio/runs/detect/nepali_htr_predict/labels
